# Paper figures: autointerp vs steering (Pythia k40, L2_w0 vs L2_w0001)

Run all cells top to bottom. Paths resolve from the repository root and expect artifacts downloaded by `scripts/download_artifact.py`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import chi2_contingency, linregress

%matplotlib inline

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def repo_root() -> Path:
    """Return the repository root containing downloaded artifact directories."""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "data_results").is_dir() or (cand / "scripts" / "download_artifact.py").is_file():
            return cand
    raise FileNotFoundError("Could not find repo root. Run this notebook from inside the repository.")


DATA = repo_root() / "data_results"

PATHS = {
    "autointerp_l2w0": DATA
    / "autointerp_all_features_l2_w0"
    / "autointerp_all_features_l2_w0"
    / "k40_l2w0_seed0_all_features_analysis.json",
    "autointerp_l2w0001": DATA
    / "autointerp_all_features_l2_w0001"
    / "autointerp_all_features_l2_w0001"
    / "k40_l2w0001_seed0_all_features_analysis.json",
    "steering_l2w0": DATA
    / "steering_all_features_results_k40_l2w0.json"
    / "steering_all_features_results_k40_l2w0.json",
    "steering_l2w0001": DATA
    / "steering_all_features_results_k40_l2w0001.json"
    / "steering_all_features_results_k40_l2w0001.json",
}

for k, p in PATHS.items():
    if not p.is_file():
        raise FileNotFoundError("Missing {}: {}".format(k, p.resolve()))

with open(PATHS["autointerp_l2w0"], "r", encoding="utf-8") as f:
    autointerp_l2w0 = json.load(f)
with open(PATHS["autointerp_l2w0001"], "r", encoding="utf-8") as f:
    autointerp_l2w0001 = json.load(f)
with open(PATHS["steering_l2w0"], "r", encoding="utf-8") as f:
    steering_l2w0 = json.load(f)
with open(PATHS["steering_l2w0001"], "r", encoding="utf-8") as f:
    steering_l2w0001 = json.load(f)

print("Data loaded.")


In [ ]:
# --- Build tables used by the three summary figures ---
df_l2w0 = pd.DataFrame(autointerp_l2w0["feature_results"])
df_l2w0["model"] = "L2_w0"

df_l2w0001 = pd.DataFrame(autointerp_l2w0001["feature_results"])
df_l2w0001["model"] = "L2_w0001"

df_steer_l2w0 = pd.DataFrame(steering_l2w0)
df_steer_l2w0["model"] = "L2_w0"
df_steer_l2w0001 = pd.DataFrame(steering_l2w0001)
df_steer_l2w0001["model"] = "L2_w0001"

SUCCESS_THRESHOLD = 4
df_steer_l2w0["success"] = df_steer_l2w0["llm_match_score"] >= SUCCESS_THRESHOLD
df_steer_l2w0001["success"] = df_steer_l2w0001["llm_match_score"] >= SUCCESS_THRESHOLD

success_rate_l2w0_sample = df_steer_l2w0["success"].mean()
success_rate_l2w0001_sample = df_steer_l2w0001["success"].mean()

contingency_sample = [
    [df_steer_l2w0["success"].sum(), len(df_steer_l2w0) - df_steer_l2w0["success"].sum()],
    [df_steer_l2w0001["success"].sum(), len(df_steer_l2w0001) - df_steer_l2w0001["success"].sum()],
]
_, p_sample_steering, _, _ = chi2_contingency(contingency_sample)

print("Steering rows: L2_w0 n={}, L2_w0001 n={}".format(len(df_steer_l2w0), len(df_steer_l2w0001)))
print(
    "Sample success (score >= {}): L2_w0 {:.1f}%, L2_w0001 {:.1f}%".format(
        SUCCESS_THRESHOLD,
        success_rate_l2w0_sample * 100,
        success_rate_l2w0001_sample * 100,
    )
)
print("Chi-square (2x2, sample-level): p = {:.4g}".format(p_sample_steering))


In [ ]:
# ==============================================================================
# THREE CLEAN SUMMARY FIGURES (paper)
# ==============================================================================

df_l2w0001_valid = df_l2w0001[df_l2w0001["explanation"] != "N/A"]
na_pct = 100 * len(df_l2w0001[df_l2w0001["explanation"] == "N/A"]) / len(df_l2w0001)

COLOR_L2W0 = "#2E86AB"
COLOR_L2W0001 = "#E94F37"

# FIGURE A
fig, ax = plt.subplots(figsize=(8, 6))
models = ["L2_w0", "L2_w0001"]
success_rates = [success_rate_l2w0_sample * 100, success_rate_l2w0001_sample * 100]
colors = [COLOR_L2W0, COLOR_L2W0001]
bars = ax.bar(models, success_rates, color=colors, edgecolor="white", linewidth=2, width=0.6)
for bar, rate in zip(bars, success_rates):
    ax.annotate(
        "{:.1f}%".format(rate),
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 14),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=26,
        fontweight="bold",
        color="black",
    )
ax.annotate("n={}".format(len(df_steer_l2w0)), xy=(0, -2), ha="center", fontsize=17, color="gray")
ax.annotate("n={}".format(len(df_steer_l2w0001)), xy=(1, -2), ha="center", fontsize=17, color="gray")
ax.set_ylabel("Steering Success Rate (%)", fontsize=24, fontweight="bold")
ax.set_title("(A) Steering Success Rate (Score \u2265 4)", fontsize=28, fontweight="bold", pad=20)
ax.set_ylim(0, max(success_rates) * 1.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", labelsize=20)
ax.tick_params(axis="y", labelsize=18)
plt.tight_layout()
plt.show()

ratio = success_rate_l2w0001_sample / success_rate_l2w0_sample
print(
    "(A) Steering: L2_w0001 is {:.1f}x the sample-level success rate of L2_w0 (chi-square p = {:.4g})".format(
        ratio, p_sample_steering
    )
)

# FIGURE B
fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0, 1, 16)
ax.hist(
    df_l2w0["autointerp_score"],
    bins=bins,
    alpha=0.7,
    label="L2_w0 (n={})".format(len(df_l2w0)),
    color=COLOR_L2W0,
    edgecolor="white",
    linewidth=1.2,
)
ax.hist(
    df_l2w0001_valid["autointerp_score"],
    bins=bins,
    alpha=0.7,
    label="L2_w0001 (n={}, {:.0f}% N/A excluded)".format(len(df_l2w0001_valid), na_pct),
    color=COLOR_L2W0001,
    edgecolor="white",
    linewidth=1.2,
)
mean_l2w0 = df_l2w0["autointerp_score"].mean()
mean_l2w0001_valid = df_l2w0001_valid["autointerp_score"].mean()
ax.axvline(mean_l2w0, color=COLOR_L2W0, linestyle="--", linewidth=3.5, alpha=0.8)
ax.axvline(mean_l2w0001_valid, color=COLOR_L2W0001, linestyle="--", linewidth=3.5, alpha=0.8)
ax.annotate(
    "\u03bc={:.2f}".format(mean_l2w0),
    xy=(mean_l2w0, ax.get_ylim()[1] * 0.96),
    fontsize=20,
    fontweight="bold",
    color=COLOR_L2W0,
    ha="center",
)
ax.annotate(
    "\u03bc={:.2f}".format(mean_l2w0001_valid),
    xy=(mean_l2w0001_valid, ax.get_ylim()[1] * 0.88),
    fontsize=20,
    fontweight="bold",
    color=COLOR_L2W0001,
    ha="center",
)
ax.set_xlabel("Autointerp Score", fontsize=24, fontweight="bold")
ax.set_ylabel("Number of Features", fontsize=24, fontweight="bold")
ax.set_title("(B) Autointerp Score Distribution", fontsize=28, fontweight="bold", pad=20)
ax.legend(fontsize=18, loc="upper left")
ax.set_xlim(0, 1.02)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="x", labelsize=18)
ax.tick_params(axis="y", labelsize=17)
plt.tight_layout()
plt.show()
print(
    "(B) Autointerp means — L2_w0: {:.2f}, L2_w0001 (valid): {:.2f}; {:.0f}% N/A excluded for L2_w0001".format(
        mean_l2w0, mean_l2w0001_valid, na_pct
    )
)

# FIGURE C
fig, ax = plt.subplots(figsize=(11, 6))
bins_c = [0, 0.6, 0.7, 0.8, 0.9, 1.01]
labels = ["<0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1.0"]
df_steer_l2w0["ai_bin"] = pd.cut(
    df_steer_l2w0["autointerp_score"], bins=bins_c, labels=labels, include_lowest=True
)
df_steer_l2w0001["ai_bin"] = pd.cut(
    df_steer_l2w0001["autointerp_score"], bins=bins_c, labels=labels, include_lowest=True
)
success_l2w0 = df_steer_l2w0.groupby("ai_bin", observed=True)["success"].mean() * 100
success_l2w0001 = df_steer_l2w0001.groupby("ai_bin", observed=True)["success"].mean() * 100
success_l2w0 = success_l2w0.reindex(labels).fillna(0)
success_l2w0001 = success_l2w0001.reindex(labels).fillna(0)
x = np.arange(len(labels))
width = 0.35
bars1 = ax.bar(
    x - width / 2,
    success_l2w0.values,
    width,
    label="L2_w0",
    color=COLOR_L2W0,
    edgecolor="white",
    linewidth=2.7,
    alpha=0.8,
)
bars2 = ax.bar(
    x + width / 2,
    success_l2w0001.values,
    width,
    label="L2_w0001",
    color=COLOR_L2W0001,
    edgecolor="white",
    linewidth=2.7,
    alpha=0.8,
)
trend_x = np.linspace(-0.5, len(labels) - 0.5, 100)
mask_l2w0 = success_l2w0.values > 0
if mask_l2w0.sum() >= 2:
    slope1, intercept1, _, _, _ = linregress(np.array(x)[mask_l2w0], success_l2w0.values[mask_l2w0])
    ax.plot(trend_x, slope1 * trend_x + intercept1, color=COLOR_L2W0, linestyle="--", linewidth=4, alpha=0.9)
mask_l2w0001 = success_l2w0001.values > 0
if mask_l2w0001.sum() >= 2:
    slope2, intercept2, _, _, _ = linregress(
        np.array(x)[mask_l2w0001], success_l2w0001.values[mask_l2w0001]
    )
    ax.plot(trend_x, slope2 * trend_x + intercept2, color=COLOR_L2W0001, linestyle="--", linewidth=4, alpha=0.9)

corr_l2w0, p_l2w0 = stats.pointbiserialr(df_steer_l2w0["success"], df_steer_l2w0["autointerp_score"])
corr_l2w0001, p_l2w0001 = stats.pointbiserialr(
    df_steer_l2w0001["success"], df_steer_l2w0001["autointerp_score"]
)
stars = "***" if p_l2w0001 < 0.001 else ""
corr_text = (
    "Correlation (Autointerp \u2192 Success):\n"
    "L2_w0:    r = {:.3f}, p = {:.4f}\n"
    "L2_w0001: r = {:.3f}, p = {:.4f}{}".format(corr_l2w0, p_l2w0, corr_l2w0001, p_l2w0001, stars)
)
ax.text(
    0.98,
    0.98,
    corr_text,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray", alpha=0.95),
    fontfamily="monospace",
)
for bar in bars1:
    if bar.get_height() > 0:
        ax.annotate(
            "{:.0f}%".format(bar.get_height()),
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
for bar in bars2:
    if bar.get_height() > 0:
        ax.annotate(
            "{:.0f}%".format(bar.get_height()),
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            fontweight="bold",
        )
ax.set_xlabel("Autointerp Score", fontsize=24, fontweight="bold")
ax.set_ylabel("Steering Success Rate (%)", fontsize=24, fontweight="bold")
ax.set_title("(C) Steering Success vs Autointerp Score", fontsize=28, fontweight="bold", pad=20)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=18)
ax.legend(fontsize=18, loc="upper left")
ax.set_ylim(0, max(max(success_l2w0), max(success_l2w0001)) * 1.35)
ax.set_xlim(-0.6, len(labels) - 0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
print("(C) Binned steering success vs autointerp score (point-biserial on raw rows).")
print("    L2_w0:    r = {:.4f}, p = {:.4f}".format(corr_l2w0, p_l2w0))
print("    L2_w0001: r = {:.4f}, p = {:.4f}".format(corr_l2w0001, p_l2w0001))
